# Streamlined CRISPR Screen Analysis Tutorial

This tutorial demonstrates the essential workflow provided by `streamlined_crispr`.
We generate a small synthetic dataset using ``benchmarking/generate_demo_dataset.py``
so that quality control, effect-size estimation, and differential expression can be
executed end-to-end without external downloads.


## Prerequisites

Install the project in editable mode (with the optional `test` extras) to make
the package importable and provide AnnData, NumPy, pandas, and SciPy:

```bash
pip install -e .[test]
```

In [1]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from benchmarking.generate_demo_dataset import write_demo_dataset
import streamlined_crispr as scr


## Prepare the demo dataset

Generate the compact `.h5ad` file if it is missing and preview it with `preview_backed` to inspect key metadata without loading the full matrix.


In [2]:
data_path = Path('../data/demo_benchmark.h5ad')
output_dir = Path('tutorial_outputs')
output_dir.mkdir(exist_ok=True)

if not data_path.exists():
    print(f'Generating demo dataset at {data_path}')
    write_demo_dataset(data_path)

adata_ro = scr.preview_backed(data_path)


AnnData object with n_obs × n_vars = 400 × 100 backed at '../data/demo_benchmark.h5ad'
    obs: 'perturbation', 'celltype'
    var: 'gene_symbols'
    uns: 'description'
First obs rows:
         perturbation celltype
cell                          
cell_000    perturb_3  NK cell
cell_001         ctrl   T cell
cell_002         ctrl   B cell
cell_003         ctrl  NK cell
cell_004    perturb_4  NK cell
First var rows:
         gene_symbols
gene                 
gene_000         G000
gene_001         G001
gene_002         G002
gene_003         G003
gene_004         G004


## Run quality control

Filtering keeps perturbations with adequate representation and removes low-quality cells and genes. When `control_label` or `gene_name_column` are omitted, the helper logs which defaults are used.


In [3]:
qc_result = scr.pp.qc_summary(
    adata_ro,
    min_genes=5,
    min_cells_per_perturbation=5,
    min_cells_per_gene=5,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial',
)
qc_result.filtered_path


PosixPath('tutorial_outputs/tutorial_filtered.h5ad')

In [4]:
qc_summary = {
    'total_cells': qc_result.cell_mask.size,
    'kept_cells': int(qc_result.cell_mask.sum()),
    'total_genes': qc_result.gene_mask.size,
    'kept_genes': int(qc_result.gene_mask.sum()),
}
qc_summary


{'total_cells': 400, 'kept_cells': 400, 'total_genes': 100, 'kept_genes': 100}

## Compute effect sizes

Both effect-size estimators stream counts from disk and persist their outputs for inspection. They reuse the inferred control label and gene identifiers gathered during QC.


In [5]:
avg_effects = scr.pb.average_log_expression(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_avg',
)
avg_effects.head()


gene,gene_000,gene_001,gene_002,gene_003,gene_004,gene_005,gene_006,gene_007,gene_008,gene_009,...,gene_090,gene_091,gene_092,gene_093,gene_094,gene_095,gene_096,gene_097,gene_098,gene_099
perturb_3,-0.276430,0.170924,-0.203878,-0.094676,-0.695081,-0.518078,0.029704,-0.484692,-0.164208,-0.417162,...,-0.685579,-0.554596,0.120910,0.113510,-0.037182,-0.708537,-0.110631,-0.348713,-0.483070,-0.308376
perturb_4,-0.274981,-0.134432,-0.074301,-0.830100,0.109846,-0.887915,-0.001699,-0.424379,0.072283,-0.135874,...,-0.063968,-0.274161,-0.241196,0.187004,-0.227663,-0.859965,-0.829832,0.213559,0.011424,-0.508053
perturb_5,-0.186417,0.307240,0.118752,-0.316685,0.390533,-0.438584,-0.135605,-0.102357,-0.393632,-0.105077,...,-0.170259,-0.037537,0.124097,-0.073591,-0.654949,-0.651327,-0.920270,-0.102471,-0.266141,-0.361187
perturb_2,-0.093631,-0.014097,0.061727,0.182905,-0.093171,0.223568,-0.164845,-0.449491,0.003060,-0.945997,...,-0.318090,-0.118704,0.053018,0.054546,0.469270,-0.620658,-0.802840,-0.130574,-0.831373,-0.527065
perturb_1,-0.462382,0.394960,-0.454084,0.270288,-0.242187,0.173952,-0.776147,3.481439,3.321851,3.304890,...,0.159154,-0.070258,0.101492,0.433829,-0.188885,-0.975481,-0.510859,-0.941034,0.048373,-0.080393


In [6]:
pseudo_effects = scr.pb.pseudobulk(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    output_dir=output_dir,
    data_name='tutorial_pseudobulk',
)
pseudo_effects.head()


gene,gene_000,gene_001,gene_002,gene_003,gene_004,gene_005,gene_006,gene_007,gene_008,gene_009,...,gene_090,gene_091,gene_092,gene_093,gene_094,gene_095,gene_096,gene_097,gene_098,gene_099
perturb_3,-0.539278,-0.317138,-0.357245,-0.419927,-0.526664,-0.532951,-0.157782,-0.385153,-0.414313,-0.271498,...,-0.361167,-0.620558,-0.351304,-0.233606,-0.386898,-0.426727,-0.348444,-0.391470,-0.501280,-0.418524
perturb_4,-0.378430,-0.470146,-0.381191,-0.719593,-0.194482,-0.632769,-0.245272,-0.559304,-0.368341,-0.244393,...,-0.262009,-0.359504,-0.409746,-0.175445,-0.387100,-0.556795,-0.498162,-0.201891,-0.269578,-0.479569
perturb_5,-0.334927,-0.190011,-0.217493,-0.440720,-0.050363,-0.625338,-0.440333,-0.263158,-0.600084,-0.351178,...,-0.200959,-0.299293,-0.242562,-0.339969,-0.594223,-0.511548,-0.654306,-0.404254,-0.367707,-0.514293
perturb_2,-0.472631,-0.405029,-0.139221,-0.307709,-0.200918,-0.226632,-0.337677,-0.435651,-0.461624,-0.656067,...,-0.198128,-0.283515,-0.300941,-0.221168,-0.128473,-0.509953,-0.599964,-0.391157,-0.656305,-0.541462
perturb_1,-0.507651,-0.215775,-0.392976,-0.231255,-0.167116,-0.368157,-0.611748,1.936391,1.821525,1.934524,...,-0.169761,-0.307200,-0.328443,-0.114736,-0.347426,-0.613212,-0.410060,-0.753514,-0.249522,-0.317254


## Differential expression

Run both the Wald and Wilcoxon tests. Each call writes an AnnData file that contains the test statistics and automatically reuses the inferred control label.


In [7]:
wald_results = scr.tl.rank_genes_groups(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    method='wald',
    output_dir=output_dir,
    data_name='tutorial_wald',
)
list(wald_results.groups)


['perturb_3', 'perturb_4', 'perturb_5', 'perturb_2', 'perturb_1']

In [8]:
wilcoxon_results = scr.tl.rank_genes_groups(
    qc_result.filtered_path,
    perturbation_column='perturbation',
    method='wilcoxon',
    output_dir=output_dir,
    data_name='tutorial_wilcoxon',
)
list(wilcoxon_results.groups)


['perturb_3', 'perturb_4', 'perturb_5', 'perturb_2', 'perturb_1']

## Inspect generated files

Every stage creates `.h5ad` artifacts for reproducibility and downstream analysis.


In [9]:
sorted(path.name for path in output_dir.glob('*.h5ad'))


['demo_avg_avg_log_effects.h5ad',
 'demo_filtered.h5ad',
 'demo_nb_nb_glm_de.h5ad',
 'demo_pseudo_pseudobulk_effects.h5ad',
 'demo_raw.h5ad',
 'demo_wald_wald_de.h5ad',
 'demo_wilcoxon_wilcoxon_de.h5ad',
 'tutorial_avg_avg_log_effects.h5ad',
 'tutorial_filtered.h5ad',
 'tutorial_pseudobulk_pseudobulk_effects.h5ad',
 'tutorial_wald_wald_de.h5ad',
 'tutorial_wilcoxon_wilcoxon_de.h5ad']

In [10]:
adata_ro.file.close()
